# Web-Based Next-Word Prediction

This notebook provides a simple web interface for the trained LSTM next-word prediction model.

The interface allows users to enter text and receive the **Top-3 predicted next words** from the trained model.

In [4]:
import json
import re
import numpy as np
import tensorflow as tf
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "technical_writing_lstm_best.keras"
)

VOCABULARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vocabulary.json"
)

CONFIG_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "model_config.json"
)

model = tf.keras.models.load_model(MODEL_PATH)

with open(VOCABULARY_PATH, "r", encoding="utf-8") as file:
    vocabulary_data = json.load(file)

word_to_index = vocabulary_data["word_to_index"]

index_to_word = {
    int(index): word
    for index, word in vocabulary_data["index_to_word"].items()
}

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    model_config = json.load(file)

SEQUENCE_LENGTH = model_config["sequence_length"]

SPECIAL_TOKENS = {
    "<PAD>",
    "<UNK>",
    "<START>",
    "<END>"
}

print("Model and configuration loaded successfully.")

Model and configuration loaded successfully.


## Next-Word Prediction Function

The following function preprocesses the user's text, prepares the required input sequence, and returns the **Top-3 most probable next words** predicted by the LSTM model.

In [5]:
def get_next_word_suggestions(text, number_of_suggestions=3):
    if not text or not text.strip():
        return []

    text = text.lower()

    tokens = re.findall(r"\b[\w']+\b", text)

    unk_index = word_to_index.get("<UNK>")

    indices = [
        word_to_index.get(token, unk_index)
        for token in tokens
    ]

    if len(indices) > SEQUENCE_LENGTH:
        indices = indices[-SEQUENCE_LENGTH:]

    pad_index = word_to_index.get("<PAD>", 0)

    if len(indices) < SEQUENCE_LENGTH:
        padding_length = SEQUENCE_LENGTH - len(indices)

        indices = (
            [pad_index] * padding_length
            + indices
        )

    input_sequence = np.array(
        indices,
        dtype=np.int32
    )

    input_sequence = np.expand_dims(
        input_sequence,
        axis=0
    )

    probabilities = model.predict(
        input_sequence,
        verbose=0
    )[0]

    sorted_indices = np.argsort(
        probabilities
    )[::-1]

    suggestions = []

    for index in sorted_indices:
        index = int(index)

        word = index_to_word.get(
            index,
            "<UNK>"
        )

        if word in SPECIAL_TOKENS:
            continue

        suggestions.append(
            (
                word,
                float(probabilities[index])
            )
        )

        if len(suggestions) >= number_of_suggestions:
            break

    return suggestions

## Launch the Web Interface

A simple Gradio interface is created below. Enter a sentence or phrase and click **Predict** to view the Top-3 next-word suggestions with their prediction probabilities.

In [6]:
import gradio as gr


def predict_words(text):
    suggestions = get_next_word_suggestions(
        text,
        number_of_suggestions=3
    )

    if not suggestions:
        return "Please enter some text."

    output = []

    for rank, (word, probability) in enumerate(
        suggestions,
        start=1
    ):
        output.append(
            f"{rank}. {word} — {probability:.2%}"
        )

    return "\n".join(output)


demo = gr.Interface(
    fn=predict_words,
    inputs=gr.Textbox(
        label="Enter Text",
        placeholder="Type your text here...",
        lines=3
    ),
    outputs=gr.Textbox(
        label="Top-3 Next-Word Suggestions",
        lines=4
    ),
    title="LSTM Next-Word Predictor",
    description=(
        "Enter text and get the three most probable "
        "next words predicted by the trained LSTM model."
    )
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset1.csv
